# **TFLite Inference Latency Benchmark & Confidence Interval**

In [ ]:
from google.colab import files

print("Upload all three .tflite files:")
uploaded = files.upload()  # upload 01, 02, 03 .tflite sekaligus

Upload all three .tflite files:


Saving 01_mobilenetv3_f16.tflite to 01_mobilenetv3_f16.tflite
Saving 02_efficientnetb0_f16.tflite to 02_efficientnetb0_f16.tflite
Saving 03_vgg16_f16.tflite to 03_vgg16_f16.tflite


# **1. Confidence Interval**

In [ ]:
from statsmodels.stats.proportion import proportion_confint

# Correct predictions locked from final evaluation run — retraining intentionally
# avoided to preserve experimental integrity (non-deterministic GPU training)
EVAL_RESULTS: dict[str, int] = {
    "MobileNetV3Small" : 41,  # 91.11%  — 41/45 correct
    "EfficientNetB0"   : 44,  # 97.78%  — 44/45 correct
    "VGG16"            : 43,  # 95.56%  — 43/45 correct
}

N = 45

print(f"{'Model':<20}  {'Accuracy':>10}  {'95% CI Lower':>13}  {'95% CI Upper':>13}")
print("─" * 62)

for model, correct in EVAL_RESULTS.items():
    acc          = correct / N
    ci_low, ci_high = proportion_confint(correct, N, alpha=0.05, method='wilson')
    print(f"  {model:<20}  {acc*100:>9.2f}%  {ci_low*100:>12.2f}%  {ci_high*100:>12.2f}%")

print("─" * 62)
print(f"  N = {N} test samples | Wilson method | alpha = 0.05")

Model                   Accuracy   95% CI Lower   95% CI Upper
──────────────────────────────────────────────────────────────
  MobileNetV3Small          91.11%         79.27%         96.49%
  EfficientNetB0            97.78%         88.43%         99.61%
  VGG16                     95.56%         85.17%         98.77%
──────────────────────────────────────────────────────────────
  N = 45 test samples | Wilson method | alpha = 0.05


# **2. Benchmark**

In [ ]:
import time
import numpy as np
import tensorflow as tf
from pathlib import Path

TFLITE_MODELS: dict[str, str] = {
    "MobileNetV3Small" : "01_mobilenetv3_f16.tflite",
    "EfficientNetB0"   : "02_efficientnetb0_f16.tflite",
    "VGG16"            : "03_vgg16_f16.tflite",
}

INPUT_SHAPE : tuple = (1, 224, 224, 3)
WARMUP_RUNS : int   = 10
BENCH_RUNS  : int   = 100


def benchmark(model_path: str, input_shape: tuple, warmup: int, runs: int) -> dict:
    """
    Measure mean and std inference latency of a TFLite model.

    Args:
        model_path:  Path to the .tflite file.
        input_shape: Input tensor shape (batch, H, W, C).
        warmup:      Number of warmup invocations before timing.
        runs:        Number of timed invocations.

    Returns:
        Dict with mean_ms and std_ms latency values.
    """
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()

    input_index = interpreter.get_input_details()[0]["index"]
    dummy_input = np.random.rand(*input_shape).astype(np.float32)
    interpreter.set_tensor(input_index, dummy_input)

    # Warmup — eliminate cold-start JIT overhead from measurements
    for _ in range(warmup):
        interpreter.invoke()

    latencies = []
    for _ in range(runs):
        start = time.perf_counter()
        interpreter.invoke()
        latencies.append(time.perf_counter() - start)

    return {
        "mean_ms" : np.mean(latencies) * 1000,
        "std_ms"  : np.std(latencies)  * 1000,
    }


print(f"{'Model':<20}  {'Mean (ms)':>10}  {'Std (ms)':>10}  {'Size (MB)':>10}")
print("─" * 58)

results = {}
for name, filename in TFLITE_MODELS.items():
    if not Path(filename).exists():
        print(f"  {name:<20}  FILE NOT FOUND — skipping")
        continue

    size_mb = Path(filename).stat().st_size / (1024 ** 2)
    stats   = benchmark(filename, INPUT_SHAPE, WARMUP_RUNS, BENCH_RUNS)

    results[name] = {**stats, "size_mb": size_mb}
    print(f"  {name:<20}  {stats['mean_ms']:>9.2f}  {stats['std_ms']:>9.2f}  {size_mb:>9.2f}")

print("─" * 58)
print(f"\nBenchmark: {BENCH_RUNS} runs per model | Warmup: {WARMUP_RUNS} runs | Input: {INPUT_SHAPE}")

Model                  Mean (ms)    Std (ms)   Size (MB)
──────────────────────────────────────────────────────────


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


  MobileNetV3Small           3.25       0.29       1.88
  EfficientNetB0            21.64       3.69       7.86
  VGG16                    388.08      59.11      28.14
──────────────────────────────────────────────────────────

Benchmark: 100 runs per model | Warmup: 10 runs | Input: (1, 224, 224, 3)
